# Step 4: Narrative retrieval

Builds a ChromaDB index over ~240k MSHA injury narratives using `sentence-transformers/all-MiniLM-L6-v2` (requires PyTorch).

**Full index build:** 15–30 minutes on CPU. **CLI:** `python -m src.tools.run_retrieval_index`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import torch
from src.tools.retrieval import EMBEDDING_MODEL_NAME, NarrativeRetriever, RETRIEVAL_META_JSON

print(f"PyTorch: {torch.__version__}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")

## Build index (skip if already built)

Set `BUILD_FULL_INDEX = True` to embed all cleaned narratives. Default checks existing index first.

In [ ]:
BUILD_FULL_INDEX = False

retriever = NarrativeRetriever()
if BUILD_FULL_INDEX or not RETRIEVAL_META_JSON.exists():
    meta = retriever.build_index(batch_size=512)
else:
    import json
    meta = json.loads(RETRIEVAL_META_JSON.read_text())
    print("Using existing index:", meta)

In [ ]:
queries = [
    "roof fall underground coal miner pinned",
    "electrical shock continuous miner cable",
    "haul truck backed into rib operator",
]

for q in queries:
    hits = retriever.search(q, top_k=2)
    print(f"\nQuery: {q}")
    for h in hits:
        print(f"  [{h.score:.3f}] doc={h.document_no}: {h.narrative[:100]}...")